In [43]:
import os
import re
import random
import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)

from torch.optim import AdamW

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

from tqdm.auto import tqdm

In [4]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cuda


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
zip_path = "/content/drive/MyDrive/dataset_sir.zip"
import zipfile

extract_path = "/content/data"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Done!")

Done!


In [7]:
train_df=pd.read_csv(f"/content/data/codemixed_train.csv")
val_df=pd.read_csv(f"/content/data/codemixed_validation.csv")
test_df=pd.read_csv(f"/content/data/codemixed_test.csv")

In [8]:
train_df.head()

,text,label
0,Ipudu atleast KCR ni jobs aduguthu thiduthunna...,0
1,Babu o rambabu nuve sa...... Naki pothuv babu,0
2,"సార్ నాదెండ్ల మనోహర్ గారు, మీరు ఎప్పటికి ప్రాణ...",1
3,Ede ithey adi RGV nijaallu Baga oppukuntaaru,1
4,ఇందుకు కాదు మిమ్మల్ని పవర్ స్టార్ అనేది పునీత్...,1


In [9]:
def clean_text(text):

    text=str(text)

    text=re.sub(r"http\S+"," ",text)

    text=re.sub(r"www\S+"," ",text)

    text=re.sub(r"@\w+"," ",text)

    text=re.sub(r"#"," ",text)

    text=re.sub(r"\s+"," ",text)

    return text.strip()

In [10]:
train_df["text"]=train_df["text"].apply(clean_text)

val_df["text"]=val_df["text"].apply(clean_text)

test_df["text"]=test_df["text"].apply(clean_text)

In [12]:
train_df=train_df.drop_duplicates(subset=["text"])

val_df=val_df.drop_duplicates(subset=["text"])

test_df=test_df.drop_duplicates(subset=["text"])

In [33]:
labels=sorted(train_df.label.unique())


In [31]:
MODEL_NAME="bert-base-multilingual-cased"

tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)

In [25]:
class TeluguDataset(Dataset):

    def __init__(self,df,tokenizer,max_len=128):

        self.texts=df.text.tolist()

        self.labels=df.label.tolist()

        self.tokenizer=tokenizer

        self.max_len=max_len

    def __len__(self):

        return len(self.texts)

    def __getitem__(self,idx):

        encoding=self.tokenizer(

            self.texts[idx],

            truncation=True,

            padding="max_length",

            max_length=self.max_len,

            return_tensors="pt"

        )

        return {

            "input_ids":encoding["input_ids"].squeeze(),

            "attention_mask":encoding["attention_mask"].squeeze(),

            "labels":torch.tensor(self.labels[idx],dtype=torch.long)

        }

In [37]:
train_dataset=TeluguDataset(train_df,tokenizer)

val_dataset=TeluguDataset(val_df,tokenizer)

test_dataset=TeluguDataset(test_df,tokenizer)

In [38]:
train_loader=DataLoader(train_dataset,batch_size=16,shuffle=True)

val_loader=DataLoader(val_dataset,batch_size=16)

test_loader=DataLoader(test_dataset,batch_size=16)

In [39]:
model=AutoModelForSequenceClassification.from_pretrained(

    MODEL_NAME,

    num_labels=2
)

model.to(device)

model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12

In [40]:
optimizer=AdamW(

    model.parameters(),

    lr=2e-5,

    weight_decay=0.01
)

In [44]:
epochs=5

total_steps=len(train_loader)*epochs

scheduler=get_linear_schedule_with_warmup(

    optimizer,

    num_warmup_steps=0,

    num_training_steps=total_steps
)

In [45]:
best_f1=0

for epoch in range(epochs):

    model.train()

    train_loss=0

    loop=tqdm(train_loader)

    for batch in loop:

        optimizer.zero_grad()

        input_ids=batch["input_ids"].to(device)

        attention=batch["attention_mask"].to(device)

        labels=batch["labels"].to(device)

        outputs=model(

            input_ids=input_ids,

            attention_mask=attention,

            labels=labels

        )

        loss=outputs.loss

        loss.backward()

        optimizer.step()

        scheduler.step()

        train_loss+=loss.item()

        loop.set_description(f"Epoch {epoch+1}")

        loop.set_postfix(loss=loss.item())

    #########################

    model.eval()

    predictions=[]

    true_labels=[]

    val_loss=0

    with torch.no_grad():

        for batch in val_loader:

            input_ids=batch["input_ids"].to(device)

            attention=batch["attention_mask"].to(device)

            labels=batch["labels"].to(device)

            outputs=model(

                input_ids=input_ids,

                attention_mask=attention,

                labels=labels

            )

            val_loss+=outputs.loss.item()

            preds=torch.argmax(outputs.logits,dim=1)

            predictions.extend(preds.cpu().numpy())

            true_labels.extend(labels.cpu().numpy())

    acc=accuracy_score(true_labels,predictions)

    p,r,f1,_=precision_recall_fscore_support(

        true_labels,

        predictions,

        average="macro"

    )

    print()

    print("Epoch:",epoch+1)

    print("Train Loss:",train_loss/len(train_loader))

    print("Val Loss:",val_loss/len(val_loader))

    print("Accuracy:",acc)

    print("Macro F1:",f1)

    if f1>best_f1:

        best_f1=f1

        torch.save(model.state_dict(),"best_model.pt")

        print("Best model saved")

  0%|          | 0/198 [00:00<?, ?it/s]


Epoch: 1
Train Loss: 0.6343918983081375
Val Loss: 0.5492713475227355
Accuracy: 0.7455919395465995
Macro F1: 0.7455854826940769
Best model saved


  0%|          | 0/198 [00:00<?, ?it/s]


Epoch: 2
Train Loss: 0.4928280957268946
Val Loss: 0.502711101770401
Accuracy: 0.7732997481108312
Macro F1: 0.7727793040293041
Best model saved


  0%|          | 0/198 [00:00<?, ?it/s]


Epoch: 3
Train Loss: 0.34381745695465743
Val Loss: 0.5756576722860336
Accuracy: 0.7405541561712846
Macro F1: 0.7348957826833933


  0%|          | 0/198 [00:00<?, ?it/s]


Epoch: 4
Train Loss: 0.20878506711486614
Val Loss: 0.635968086719513
Accuracy: 0.7682619647355163
Macro F1: 0.7682252005279724


  0%|          | 0/198 [00:00<?, ?it/s]


Epoch: 5
Train Loss: 0.13169758527473818
Val Loss: 0.7667308139801026
Accuracy: 0.7682619647355163
Macro F1: 0.767018779342723


In [46]:
model.load_state_dict(torch.load("best_model.pt"))

model.eval()

predictions=[]

true_labels=[]

with torch.no_grad():

    for batch in test_loader:

        input_ids=batch["input_ids"].to(device)

        attention=batch["attention_mask"].to(device)

        labels=batch["labels"].to(device)

        outputs=model(

            input_ids=input_ids,

            attention_mask=attention

        )

        preds=torch.argmax(outputs.logits,dim=1)

        predictions.extend(preds.cpu().numpy())

        true_labels.extend(labels.cpu().numpy())

In [47]:
acc=accuracy_score(true_labels,predictions)

precision,recall,f1,_=precision_recall_fscore_support(

    true_labels,

    predictions,

    average="macro"

)

print(f"Accuracy : {acc:.4f}")

print(f"Precision: {precision:.4f}")

print(f"Recall   : {recall:.4f}")

print(f"Macro F1 : {f1:.4f}")

Accuracy : 0.7725
Precision: 0.7729
Recall   : 0.7716
Macro F1 : 0.7719


In [49]:
print(classification_report(
    true_labels,
    predictions,
    target_names=["Class 0", "Class 1"]
))
cm=confusion_matrix(true_labels,predictions)

print(cm)

              precision    recall  f1-score   support

     Class 0       0.78      0.74      0.76       194
     Class 1       0.77      0.80      0.78       206

    accuracy                           0.77       400
   macro avg       0.77      0.77      0.77       400
weighted avg       0.77      0.77      0.77       400

[[144  50]
 [ 41 165]]
